# Slow-Light Black Hole Imaging from GRMHD Simulations

In this notebook, we move beyond the fast-light approximation and explore slow-light radiative transfer. 
This approach accounts for the time-dependent evolution of the plasma, enabling more physically consistent 
black hole images derived from GRMHD simulations.

- In Jipole, this is done by enabling the variable SLOW_LIGHT = true

In [1]:
using Jipole
using StaticArrays

const MBH = 6.2e9

6.2e9

We must account for the time evolution of the plasma during photon propagation. In practice, this means that a single simulation snapshot is no longer sufficient.

Instead, slow-light radiative transfer requires access to multiple GRMHD dump files, each corresponding to a different simulation time. As photons travel through the domain, their trajectories intersect the simulation at different times, and the local fluid quantities (e.g., density, temperature, magnetic field) must be interpolated from the appropriate dumps.

In this notebook, we therefore work with a sequence of simulation outputs stored as:


In [2]:
const all_dumps_path = "/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.%05d.h5"

"/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.%05d.h5"

To perform slow-light, we must specify both the temporal range of the simulation data and how frequently images are produced. 

The parameter `dump_max` sets the extent of the available simulation snapshots, meaning that the radiative transfer will interpolate fluid quantities using dumps from the initial index up to this maximum value. 

In this example, the evolution of the plasma is sampled from dump 0 through dump 10, providing a finite time window over which photon trajectories are evaluated.

The parameter `ImageCadence` controls how often images are generated in physical time. For instance, a value of 10 corresponds to producing one image every \(10\ r_g / c\), where \(r_g\) is the gravitational radius. This defines the temporal resolution of the resulting sequence of images and ultimately determines how smoothly the time variability of the system is captured.


All of this information is stored in an `OfSlowLight` structure, which manages how simulation data is loaded and interpolated in time during the ray-tracing process.

At any given moment, only a small subset of the available dumps is kept in memory to remain efficient. In this implementation, we always keep three dump files loaded. These define a sliding time window over which interpolation is performed. The variables `tA` and `tB` correspond to the times of two consecutive dumps (for example, dump 0 and dump 1). If a photon intersects the simulation at a time between `tA` and `tB`, the fluid quantities are obtained by interpolating between these two snapshots. As the photon propagates forward in time and moves beyond `tB`, the window shifts, and interpolation is then performed between the next pair of dumps (e.g., dumps 1 and 2, then 2 and 3, and so on).

The field `current_dumps_path` keeps track of the next dump file to be loaded into memory as this sliding window advances. This allows the code to progressively stream simulation data without loading all dumps at once.

`tf` represents the final time of the simulation data, corresponding to the last available dump as defined by `dump_max`. This sets the upper temporal boundary for the slow-light calculation.

In [3]:
const dump_min = 200
const dump_max = 2000
const ImageCadence = 0.5 
params_slowlight = Jipole.Slowlight.OfSlowLight(dump_min, dump_max, ImageCadence, 0.0, 0.0, 0.0, "")
params_slowlight.current_dumps_path = Jipole.Slowlight.update_dump_path(params_slowlight, all_dumps_path)

"/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.00200.h5"

In [4]:
trat_large = 20. 
const trat_small = 1. 
const beta_crit = 1.0 
const th_beg = 1.74e-2 
const sigma_cut = 1.0 
const sigma_cut_high = -1.0;

### Choosing `M_unit`

`M_unit` sets the overall density/temperature normalization for the GRMHD dump, and therefore the entire radiative transfer calculation. Always pass it explicitly rather than relying on `read_header`'s default.

For the M87*-scale dumps used in this repo (`MBH = 6.2e9 Msun`, `D = 16.9 Mpc`), the two common GRMHD magnetic topologies use:

- **MAD** (Magnetically Arrested Disk): `M_unit = Jipole.Constants.M_UNIT_MAD` (`6.e24` g)
- **SANE** (Standard And Normal Evolution): `M_unit = Jipole.Constants.M_UNIT_SANE` (`3.e26` g -- `read_header`'s default)

In [5]:
const model = Jipole.Iharm.read_header(params_slowlight.current_dumps_path, MBH;
    th_beg=th_beg, trat_small=trat_small, beta_crit=beta_crit,
    sigma_cut=sigma_cut, sigma_cut_high=sigma_cut_high, slow_light=true,
    M_unit=Jipole.Constants.M_UNIT_MAD);

Initializing grid from: /work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.00200.h5


custom electron model loaded from dump file...
Using Modified Kerr-Schild coordinates MKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.165908 Rout: 1000.000000
Grid start (startx): 1.535001367961415e-01, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 2.638380914916404e-02, 7.812500000000000e-03, 4.908738521234052e-02


We begin by initializing a vector of simulation data containing three dump files and loading the first three snapshots into memory. These initial dumps define the starting interpolation window for the slow-light calculation.

As photon trajectories advance in time, this window must be updated to ensure that the appropriate simulation data is always available. All loading of new dump files and shifting of the sliding window is handled internally by the `update_data!` function (defined in `slowlight.jl`). This routine takes care of discarding old snapshots, loading new ones, and maintaining the correct ordering for time interpolation throughout the radiative transfer.

In [7]:
const simulation_data = Vector{Jipole.Iharm.IharmData{Float64,Array{Float64,3},Float64,Array{Float64,3}}}(undef, 3)
# Every time a dump is loaded with `advance_path!` set, slow light mode
# automatically advances to the next one.
advance! = () -> (params_slowlight.current_dumps_path = Jipole.Slowlight.update_dump_path(params_slowlight, all_dumps_path))
simulation_data[1] = Jipole.Iharm.load_data(params_slowlight.current_dumps_path, trat_large, model; advance_path! = advance!)
simulation_data[2] = Jipole.Iharm.load_data(params_slowlight.current_dumps_path, trat_large, model; advance_path! = advance!)
simulation_data[3] = Jipole.Iharm.load_data(params_slowlight.current_dumps_path, trat_large, model; advance_path! = advance!)

params_slowlight.tA = simulation_data[1].t;
params_slowlight.tB = simulation_data[2].t;

params_slowlight.tf = Jipole.Slowlight.get_specific_dump_time(params_slowlight.dump_max, all_dumps_path);

Loading data from '/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.00200.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (256, 128, 128)
Loading data from '/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.00201.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (256, 128, 128)
Loading data from '/work/hdd/bekt/GRMHD/LowResJipoleConvertedToH5/torus.out0.00202.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (256, 128, 128)


In [8]:
# Observer distance in gravitational radii (Rg)
const ro = 1000.0

# Inclination angle (deg) — angle between the observer and the BH spin axis
const th = 29.0

# Azimuthal angle (deg) — rotation around the system
const phi = 0.0

# Image resolution — total geodesics traced = pixels_x * pixels_y
const pixels_x = 512
const pixels_y = 512

# Distance to the source (in parsecs, converted to code units)
const SourceD = 16.9e6 * Jipole.Constants.PC

# Event horizon radius for a Kerr black hole
const Rh = 1 + sqrt(1. - model.a * model.a)

# Observing frequency (Hz), e.g. 230 GHz for EHT-like images
const freq = 230e9

# Image plane size (in Rg), scaled from physical distance
const DXsize = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160
const DYsize = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160

# Field of view (radians)
const fovx = DXsize / ro
const fovy = DYsize / ro

# Image offsets (can be used to shift the camera)
const xoff = 0.0
const yoff = 0.0

const maxnstep = 15000

15000

## Geodesic Ray-Tracing and Time Boundary Estimation

In this stage, we perform the core ray-tracing calculation to construct the final image from the GRMHD simulation. The camera position is first defined in the native coordinate system of the spacetime, and the observed frequency is converted into a dimensionless unit system appropriate for the geodesic integration.

We allocate memory for the auxiliary arrays to track the number of integration steps per pixel and the number of midplane crossings. Because the computation is parallelized across multiple threads, each thread is assigned its own scratch space to store geodesic trajectories. Each trajectory is represented as a sequence of spacetime points and wavevectors, with a fixed maximum number of integration steps to ensure memory safety.

Following the conventions used in ipole, we compute two diagnostic time boundaries: one corresponding to the first entrance into the emission region and another corresponding to the last relevant interaction before the photon escapes. These quantities are tracked per thread and later reduced across all threads to obtain global extrema. In addition, we record the absolute minimum photon time encountered across all rays, which defines the earliest emission time needed for the simulation.

After all pixels have been processed, the intensity map is scaled by the appropriate frequency factor, completing the radiative transfer step.

Finally, temporary arrays storing thread-local geodesics and timing information are discarded, and garbage collection is triggered to free memory before proceeding to the next stage of the calculation. This ensures that only the necessary global quantities are retained for subsequent slow-light processing.

In [9]:
using ProgressMeter

# Calculate the camera position in native coordinates
Xcamera = MVector{4,Float64}(Jipole.Camera.camera_position(ro, th, phi, model.a, model))

# Unitless frequency 
const freq_unitless = freq * Jipole.Constants.HPL / (Jipole.Constants.ME * Jipole.Constants.CL * Jipole.Constants.CL)

# Array that will hold the Intensity value for each pixel
midplane_crossings = zeros(Int, pixels_x, pixels_y)
nsteps = zeros(Int, pixels_x, pixels_y)

# Number of threads used in the calculation
println("Allocating workspaces for $pixels_x row-tasks...")
dummy_svec = @SVector zeros(4)
dummy_traj = Jipole.GeoTypes.OfTrajS(0.0, dummy_svec, dummy_svec, dummy_svec, dummy_svec)


task_trajs = [Vector{Jipole.GeoTypes.OfTrajS}(undef, maxnstep) for _ in 1:pixels_x]
for i in 1:pixels_x
    for k in 1:maxnstep
        task_trajs[i][k] = dummy_traj
    end
end


# This will hold the exact number of steps for each pixel.
const all_geodesics = Matrix{Vector{Jipole.GeoTypes.OfTrajS}}(undef, pixels_x, pixels_y)
row_t0 = zeros(Float64, pixels_x)
row_tgeoi = fill(-1e100, pixels_x)
row_tgeof = zeros(Float64, pixels_x)

# 2. Reintroduce the atomic counter
pixels_processed = Threads.Atomic{Int}(0)
total_pixels = pixels_x * pixels_y

p = Progress(
    total_pixels; 
    desc = "Raytracing Image...", 
    showspeed = true, 
    barlen = 30
)
ProgressMeter.ijulia_behavior(:clear)

println("Tracing Geodesics...")
Threads.@threads for i in 0:(pixels_x - 1)
    my_traj = task_trajs[i + 1]
    
    for j in 0:(pixels_y - 1)
        nstep, midplane_crossings[i+1,j+1] = Jipole.Geodesics.get_pixel(
            my_traj, i, j, Xcamera, 
            fovx, fovy, freq_unitless, 
            pixels_x, pixels_y, model.a, 
            Rh, model.rmax_geo, model, xoff, yoff
        ) 
        nsteps[i+1, j+1] = nstep
        
        # Save to permanent storage
        all_geodesics[i + 1, j + 1] = my_traj[1:nstep]

        final_step_time = my_traj[nstep].X[1]
        
        # 3. Index using i+1 instead of the unsafe tid
        if final_step_time < row_t0[i+1]
            row_t0[i+1] = final_step_time
        end

        # tgeoi and tgeof calculations
        pixel_tgeoi = 1.0
        pixel_tgeof = 1.0
        for k in 1:nstep
            X = my_traj[k].X
            K = my_traj[k].Kcon
        
            log_r = X[2]
            t_coord = X[1]
            k_r = K[2]
        
            if pixel_tgeoi > 0.0 && log_r < log(100.0)
                pixel_tgeoi = t_coord
            end
        
            if pixel_tgeof > 0.0 && log_r > log(100.0) && k_r < 0.0
                pixel_tgeof = t_coord
            end
        end
        final_step_time = my_traj[nstep].X[1]
        
        # Index using i+1
        if pixel_tgeoi < 0.0 && pixel_tgeoi > row_tgeoi[i+1]
            row_tgeoi[i+1] = pixel_tgeoi
        end
        if pixel_tgeof < 0.0 && pixel_tgeof < row_tgeof[i+1]
            row_tgeof[i+1] = pixel_tgeof
        elseif pixel_tgeof > 0.0 && final_step_time < row_tgeof[i+1]
            row_tgeof[i+1] = final_step_time
        end
        
        # 4. Use the optimized progress update block
        current_count = Threads.atomic_add!(pixels_processed, 1) + 1
        
        if current_count % 2000 == 0 || current_count == total_pixels
            ProgressMeter.update!(
                p, current_count; 
                showvalues = [
                    (:pixel, "($i, $j)"), 
                    (:total_done, "$current_count/$total_pixels")
                ]
            )
        end
    end
end

finish!(p);

# Now calculate minimums from the row arrays instead of thread arrays
t0 = minimum(row_t0)
tgeof = minimum(row_tgeof) 
tgeoi = maximum(row_tgeoi) 

println("Calculated t0 (absolute longest time): $t0")
println("Calculated tgeof (oldest active time): $tgeof")
println("Calculated tgeoi (newest active time): $tgeoi")

row_t0 = nothing
row_tgeoi = nothing
row_tgeof = nothing
GC.gc()

Raytracing Image... 100%|██████████████████████████████| Time: 0:03:11 ( 0.73 ms/it)
        pixel: (255, 511)
   total_done: 262144/262144


Calculated t0 (absolute longest time): -1182.1119279877755
Calculated tgeof (oldest active time): -1182.1119279877755
Calculated tgeoi (newest active time): -909.3451036995951


## Slow-Light Image Construction and Radiative Transfer

The function `process_slowlight_images!` performs the final stage of the slow-light pipeline.

The procedure begins by determining the first image time relative to the simulation window, using the earliest geodesic interaction time and the slow-light configuration. Based on the photon propagation times and the chosen image cadence, a set of image frames is scheduled across the available temporal domain. Because multiple images may be active simultaneously, a buffer of image structures is allocated to hold partially computed frames.

Each image corresponds to a specific target emission time. For every pixel in the image, the precomputed geodesic trajectory is reused and integrated backwards in time through the evolving simulation. Along each trajectory, the code steps through spacetime points and reconstructs the radiative transfer by sampling the local plasma properties at the appropriate emission time. This requires dynamically adjusting the photon time coordinate to remain within the valid slow-light time window defined by the loaded simulation dumps.

Once all pixels of a frame are processed, the resulting intensity map is scaled by the appropriate frequency factor and written to disk as a time-stamped output file. Frames that are not yet complete remain in memory until sufficient simulation data has been loaded to finalize them.

After each batch of frames, the simulation window is advanced using `update_data!`, which loads the next GRMHD dump and shifts the slow-light interpolation window forward in time. This ensures that photon trajectories always have access to the correct plasma state as they traverse the evolving simulation.

The loop continues until all images have been fully constructed and no active frames remain, producing a complete time-dependent movie of the slow-light black hole emission.

In [9]:
const scale_factor = Jipole.Imaging.calculate_scale_factor(DXsize, DYsize, pixels_x, pixels_y, SourceD, model.L_unit)

0.2295350232600494

In [ ]:
Jipole.Slowlight.process_slowlight_images!(
    params_slowlight, 
    simulation_data, 
    all_geodesics, 
    nsteps, 
    model, 
    t0, 
    tgeof, 
    tgeoi, 
    pixels_x, 
    pixels_y, 
    freq,
    trat_large,
    all_dumps_path,
    Xcamera,
    ro,
    th, 
    phi,
    fovx,
    fovy,
    SourceD,
    scale_factor
)